<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

# 컴퓨터과학 예제: 작업 배정

## 1) 예제 설명

- 네 작업을 두 컴퓨터에 배정하여 전체 완료 시간 최소화
- 두 컴퓨터의 속도와 시작 시점 동일, 각 컴퓨터는 작업을 순차 처리, 작업 분할과 통신 시간은 고려하지 않음

| 작업 | A | B | C | D |
|:---|:---|:---|:---|:---|
| 처리 시간 (초) | 8 | 7 | 6 | 5 |

- 의사결정벡터(decision vector): $\mathbf{x}=(x_1,x_2,x_3,x_4)$, 각 성분은 A·B·C·D의 담당 컴퓨터 번호 0 또는 1
- 입자 위치: 탐색에 사용하는 실수 벡터 $\mathbf{z}=(z_1,z_2,z_3,z_4)$, 각 성분의 범위는 0~1
- 부하(load): 컴퓨터별 작업 시간의 합
- 목적함수(objective function): 두 부하 중 큰 값인 전체 완료 시간, 작은 값이 좋은 결과

- A·B와 C·D로 배정하면 부하는 15초와 11초, 전체 완료 시간은 15초


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 2) 구현

- 공통 PSO 함수는 탐색 후 전역 최적 위치 gbest 반환

- 초기 속도의 각 성분은 해당 탐색 구간 폭의 −10%~10%에서 균등분포 난수로 설정


In [ ]:
import numpy as np


def particle_swarm(initial_positions, objective, lower, upper, iterations=20, seed=0):
    # 난수 생성기와 초기 위치 설정
    rng = np.random.default_rng(seed)
    positions = np.array(initial_positions, dtype=float, copy=True)
    if np.any(positions < lower) or np.any(positions > upper):
        raise ValueError("Initial positions must satisfy the bounds.")

    # 탐색 구간 폭에 비례하는 초기 속도 설정
    span = np.asarray(upper) - np.asarray(lower)
    velocities = rng.uniform(-0.1, 0.1, positions.shape) * span

    # 초기 위치를 평가하여 개인 및 전역 최적 위치 설정
    pbest = positions.copy()
    best_values = np.array([objective(x) for x in positions])
    gbest = pbest[np.argmin(best_values)].copy()

    for _ in range(iterations):
        # 관성항, 인지항, 사회항으로 속도 갱신
        r1 = rng.random(positions.shape)
        r2 = rng.random(positions.shape)
        inertia = 0.6 * velocities
        cognitive_term = 1.4 * r1 * (pbest - positions)
        social_term = 1.4 * r2 * (gbest - positions)
        velocities = inertia + cognitive_term + social_term

        # 위치 이동 후 경계값으로 제한, 벗어난 성분의 속도는 0으로 설정
        proposed = positions + velocities
        positions = np.clip(proposed, lower, upper)
        velocities[positions != proposed] = 0.0

        # 이동한 위치의 목적함수값 계산
        values = np.array([objective(x) for x in positions])

        # 기존 기록보다 좋은 위치로 개인 최적 위치 갱신
        improved = values < best_values
        pbest[improved] = positions[improved]
        best_values[improved] = values[improved]

        # 모든 개인 최적 위치 중 가장 좋은 위치 선택
        gbest = pbest[np.argmin(best_values)].copy()

    # 반복 종료 후 전역 최적 위치 반환
    return gbest


<style>
ul { padding-left:24px; }
table { margin-left:24px !important; margin-right:0 !important; }
th, td { text-align:left !important; }
li table { margin-left:0 !important; }
</style>

## 3) 예제

- 공통 PSO 함수에 작업 시간과 배정 변환·평가 함수를 전달, 20회 반복 후 작업 배정·부하·완료 시간 출력


In [ ]:
# 작업별 처리 시간
durations = [8, 7, 6, 5]

# 실수 위치를 컴퓨터 배정 번호로 변환
def decode(position):
    return (np.asarray(position) >= 0.5).astype(int)

# 컴퓨터별 부하와 전체 작업의 완료 시간 계산
def evaluate(decision):
    loads = [0, 0]
    for i in range(4):
        loads[decision[i]] += durations[i]
    return {"loads": loads, "completion_time": max(loads)}

# 두 컴퓨터의 부하 중 큰 값을 최소화
def objective(position):
    return evaluate(decode(position))["completion_time"]

# 초기 후보 설정 및 PSO 실행
initial_positions = [[0, 0, 0, 0], [1, 1, 1, 1],
                     [0, 0, 1, 1], [1, 1, 0, 0]]
result = particle_swarm(initial_positions, objective, 0, 1, iterations=20, seed=2)

# 탐색 결과를 실제 의사결정으로 변환하고 출력
decision = decode(result)
evaluation = evaluate(decision)
print("Decision:", decision)
print("Loads (s):", evaluation["loads"])
print("Completion time (s):", evaluation["completion_time"])
print("Stop reason: iteration limit")
